# Version 2.1 - Complete Embedding

In [ ]:
%pip install langchain langchain-text-splitters langchain-chroma langchain-openai langchain-community langchain-huggingface langchain-classic crewai crewai-tools sentence-transformers ipywidgets pypdf tqdm ansi2html

Note: you may need to restart the kernel to use updated packages.


In [10]:
%pip install jupyter-resource-usage

Note: you may need to restart the kernel to use updated packages.


In [3]:
!free -h

               total        used        free      shared  buff/cache   available
Mem:           123Gi        92Gi        16Gi        68Mi        14Gi        30Gi
Swap:          884Gi       116Gi       768Gi


In [1]:
import os
import json
from pathlib import Path
from langchain_chroma import Chroma
from langchain_text_splitters import RecursiveCharacterTextSplitter
from crewai.tools import tool
from langchain_classic.storage import LocalFileStore
from langchain_classic.embeddings import CacheBackedEmbeddings
from langchain_community.document_loaders import DirectoryLoader, TextLoader, PyPDFLoader
from langchain_huggingface import HuggingFaceEmbeddings
import logging
import datetime
import shutil
import pickle
import hashlib

In [2]:
import psutil

def get_resources():
    """
    Returns current CPU and RAM usage in a clean format.
    Very lightweight - uses almost no resources.
    """
    # CPU usage
    cpu_percent = psutil.cpu_percent(interval=0.1)   # interval=0.1 is fast and accurate enough
    
    # RAM usage
    ram = psutil.virtual_memory()
    
    return {
        "CPU": f"{cpu_percent:.1f}%",
        "RAM": f"{ram.percent:.1f}%",
        "RAM_Used": f"{ram.used / (1024**3):.2f} GB",
        "RAM_Total": f"{ram.total / (1024**3):.2f} GB",
        "RAM_Available": f"{ram.available / (1024**3):.2f} GB"
    }


# Usage
resources = get_resources()
print(resources)

{'CPU': '66.2%', 'RAM': '97.9%', 'RAM_Used': '121.14 GB', 'RAM_Total': '123.79 GB', 'RAM_Available': '2.65 GB'}


In [3]:
def setup_logger(log_dir, log_name, logger_obj_name="logger_obj_name"):
    """
    Configure the logger system: output to both console and file simultaneously.
    """
    # 1. If the log directory doesn't exist, create it
    if not os.path.exists(log_dir):
        os.makedirs(log_dir)
        print(f"Log directory created: {log_dir}")

    # 2. Generate a timestamped log filename, e.g., scraper_log_20231027_103000.log
    log_filename = f"{log_name}-{datetime.datetime.now().strftime('%Y%m%d-%H%M%S')}.log"
    log_filepath = os.path.join(log_dir, log_filename)

    # 3. Create Logger object
    logger = logging.getLogger(logger_obj_name)
    logger.setLevel(logging.INFO) # Set the minimum logger level
    logger.handlers = [] # Clear previous handlers to prevent duplicate printing

    # --- Define a unified format (time accurate to the second) ---
    # %(asctime)s : Time
    # %(levelname)s : Log level (INFO/ERROR)
    # %(message)s : Your message content
    file_formatter = logging.Formatter(
        '[%(asctime)s][%(levelname)s] %(message)s', 
        datefmt="%y-%#m-%#d %H:%M:%S"
    )
    console_formatter = logging.Formatter(
        '[%(asctime)s][%(levelname)s] %(message)s', 
        datefmt="%y-%#m-%#d %H:%M:%S"
    )

    # --- Handler 1: File output (detailed, with timestamp) ---
    file_handler = logging.FileHandler(log_filepath, encoding='utf-8')
    file_handler.setFormatter(file_formatter)
    logger.addHandler(file_handler)

    # --- Handler 2: Console output (concise, for human reading) ---
    console_handler = logging.StreamHandler()
    console_handler.setFormatter(console_formatter)
    logger.addHandler(console_handler)

    logger.info(f"✅ logger System Started")
    logger.info(f"Log file path: {log_filepath}")
    return logger

In [4]:
# ====================== CONFIG: EVERYTHING ON YOUR OTHER DRIVE ======================
BASE_DIR = Path.cwd() # current directory
# WQB_FORUM_PATH = BASE_DIR / "Docs" / "Forums"
WQB_FORUM_CHINA_PATH = BASE_DIR / "Docs" / "Forums" / "wqb_china_consultant_pdf"
WQB_FORUM_GLOBAL_PATH = BASE_DIR / "Docs" / "Forums" / "wqb_global_consultant_pdf"
WQB_FORUM_RESEARCH_PATH = BASE_DIR / "Docs" / "Forums" / "wqb_research_pdf"
WQB_FORUM_TIPS_PATH = BASE_DIR / "Docs" / "Forums" / "wqb_brain_tips_pdf"
WQB_OFFICIAL_DOCS_PATH = BASE_DIR / "Docs" / "OfficialDocs"
# Note: PaymentPolicy store in WQB_FORUM_TIPS_PATH since it has only few pdfs
# WQB_PAYMENT_POLICY_PATH = BASE_DIR / "Docs" / "PaymentPolicy"

EMBEDDING_DB_FORUM_CHINA_DIR = BASE_DIR / "embedding_db" / "wqb_forum_china_embedding_db"
EMBEDDING_DB_FORUM_GLOBAL_DIR = BASE_DIR / "embedding_db" / "wqb_forum_global_embedding_db"
EMBEDDING_DB_FORUM_RESEARCH_DIR = BASE_DIR / "embedding_db" / "wqb_forum_research_embedding_db"
EMBEDDING_DB_FORUM_TIPS_DIR = BASE_DIR / "embedding_db" / "wqb_forum_tips_embedding_db"
EMBEDDING_DB_OFFICIALDOCS_DIR = BASE_DIR / "embedding_db" / "wqb_official_docs_embedding_db"
HF_CACHE_DIR = BASE_DIR / "cache" / "hf"
LOG_DIR = BASE_DIR / "logs" / datetime.datetime.now().strftime("%Y%m")

# Create the folders (pathlib makes this easy too)
HF_CACHE_DIR.mkdir(parents=True, exist_ok=True)
EMBEDDING_DB_FORUM_CHINA_DIR.mkdir(parents=True, exist_ok=True)
EMBEDDING_DB_FORUM_GLOBAL_DIR.mkdir(parents=True, exist_ok=True)
EMBEDDING_DB_FORUM_RESEARCH_DIR.mkdir(parents=True, exist_ok=True)
EMBEDDING_DB_FORUM_TIPS_DIR.mkdir(parents=True, exist_ok=True)
EMBEDDING_DB_OFFICIALDOCS_DIR.mkdir(parents=True, exist_ok=True)
LOG_DIR.mkdir(parents=True, exist_ok=True)

# ====================== INITIALIZE LOGGER ======================
logger = setup_logger(LOG_DIR, "wqb_agent", "wqb_main_logger")

# Loader Dict
LOADER_DICT = {
    "pdf": PyPDFLoader,
    "txt": TextLoader
}

[26-05-11 15:55:01][INFO] ✅ logger System Started
[26-05-11 15:55:01][INFO] Log file path: /data/logs/202605/wqb_agent-20260511-155501.log


In [5]:
embeddings = HuggingFaceEmbeddings(
    model_name="BAAI/bge-m3", # excellent for Chinese
    cache_folder=str(HF_CACHE_DIR),  # Use the custom cache directory
    model_kwargs={"device": "cpu"},           # force CPU (your low GPU setup)
    encode_kwargs={"normalize_embeddings": True},  # best for Chroma similarity search
    show_progress=True
)

Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

In [6]:
# Usage
resources = get_resources()
print(resources)

{'CPU': '74.4%', 'RAM': '99.0%', 'RAM_Used': '122.61 GB', 'RAM_Total': '123.79 GB', 'RAM_Available': '1.18 GB'}


In [7]:
# ==================================== Ingest PDFs (ONE-TIME) ====================================
def ingest_embeddings_no_cache(docs_path, embeddings, vectorstore_dir, docs_type, source_type, batch_size=1000):
    # Define where you want to save the cache
    CHUNK_CACHE_PATH = os.path.join(BASE_DIR, f"chunks_cache_{source_type}.pkl")
    docs_path_abs = os.path.abspath(docs_path)
    vectorstore_dir_abs = os.path.abspath(vectorstore_dir)
    ingest_state_path = os.path.join(docs_path_abs, "ingested_files.json")

    logger.info(f"Starting {source_type} ingestion from: {docs_path}")
    
    # ==========================================
    # 1. ATTEMPT TO LOAD THE CACHE
    # ==========================================
    cache_loaded = False
    if os.path.exists(CHUNK_CACHE_PATH):
        logger.info(f"📦 Found local cache for {source_type}! Loading from disk...")
        try:
            with open(CHUNK_CACHE_PATH, "rb") as f:
                cache_data = pickle.load(f)
            
            # Extract our variables from the cache
            splits = cache_data["splits"]
            newly_added_files = cache_data["newly_added_files"]
            already_ingested = cache_data["already_ingested"]
            
            logger.info(f"✅ Successfully loaded {len(splits)} chunks from cache. Skipping PDF parsing!")
            cache_loaded = True
        except Exception as e:
            logger.warning(f"⚠️ Failed to load cache: {e}. Will parse files from scratch.")
            cache_loaded = False

    # ==========================================
    # 2. PARSE AND SPLIT (Only if cache is empty/failed)
    # ==========================================
    if not cache_loaded:
        if docs_type not in LOADER_DICT:
            logger.error(f"Unsupported document type: {docs_type}. Supported types are: {list(LOADER_DICT.keys())}")
            return False
    
        # State is tied to vectorstore_dir. If DB path changes, treat as a fresh DB.
        ingest_state = {
            "vectorstore_dir": vectorstore_dir_abs,
            "files": []
        }
    
        if os.path.exists(ingest_state_path):
            try:
                with open(ingest_state_path, "r", encoding="utf-8") as f:
                    existing_state = json.load(f)
                if isinstance(existing_state, dict) and existing_state.get("vectorstore_dir") == vectorstore_dir_abs:
                    ingest_state = {
                        "vectorstore_dir": vectorstore_dir_abs,
                        "files": existing_state.get("files", []) if isinstance(existing_state.get("files", []), list) else []
                    }
                else:
                    logger.info("Detected a new embedding DB path. Resetting ingest tracking state.")
            except Exception as e:
                logger.warning(f"Failed to read ingest state file, starting fresh. Reason: {e}")
    
        already_ingested = set(ingest_state["files"])
    
        loader = DirectoryLoader(
            str(docs_path),
            glob=f"**/*.{docs_type}",
            loader_cls=LOADER_DICT.get(docs_type),
            silent_errors=False,
            show_progress=True
        )
    
        logger.info("Loading documents (this may take a while)...")
        try:
            docs = loader.load()
            logger.info(f"Successfully loaded {len(docs)} {docs_type.upper()} files.")
        except Exception as e:
            logger.error(f"❌ Failed to load {docs_type.upper()}s: {e}", exc_info=True)
            return False
    
        docs_to_ingest = []
        newly_added_files = set()
        # for doc in docs:
        #     source_file = os.path.abspath(doc.metadata.get("source", ""))
        #     if source_file and source_file in already_ingested:
        #         continue
        #     docs_to_ingest.append(doc)
        #     if source_file:
        #         newly_added_files.add(source_file)
    
        # ============ Fix: Use relative path record ===============
        for doc in docs:
            raw_source = doc.metadata.get("source", "")
            
            if raw_source:
                # 1. Get the absolute path of the file
                abs_source = os.path.abspath(raw_source)
                
                # 2. Find the relative path from your project root (BASE_DIR)
                # This turns '/data/Docs/OfficialDocs/file.pdf' into 'Docs/OfficialDocs/file.pdf'
                # Docs/OfficialDocs/file.pdf is better than /Docs/OfficialDocs/file.pdf
                rel_source = os.path.relpath(abs_source, start=BASE_DIR)
                
                # 3. Force forward slashes for Windows/Linux cross-compatibility
                source_file = rel_source.replace("\\", "/")
            else:
                source_file = ""
                
            if source_file and source_file in already_ingested:
                continue
                
            docs_to_ingest.append(doc)
            
            if source_file:
                newly_added_files.add(source_file)
        
        if not docs_to_ingest:
            logger.info("No new files to ingest. All matching files already exist in ingest state.")
            return True
    
        logger.info(f"New files to ingest: {len(docs_to_ingest)}")
    
        logger.info("Splitting documents into chunks...")
        text_splitter = RecursiveCharacterTextSplitter(chunk_size=2000, chunk_overlap=400)
        splits = text_splitter.split_documents(docs_to_ingest)
        logger.info(f"Created {len(splits)} text chunks.")
    
        # Add source metadata to each chunk for better traceability if source_type is provided
        if source_type:
            for chunk in splits:
                chunk.metadata["source_type"] = source_type
            logger.info(f"Added source_type metadata to each chunk: {source_type}")
    
        # ============ FIX: Sanitize Metadata ============
        # PyPDFLoader sometimes extracts complex objects (like nested dictionaries, empty lists, or None values). 
        # Chroma's strict Rust/SQLite backend crashes violently if it encounters anything other than a string, 
        # integer, float, or boolean in the metadata. The Fix: Added a loop before the database insertion 
        # that strips out any invalid data types.
        logger.info("Sanitizing metadata...")
        for chunk in splits:
            clean_metadata = {}
            for key, value in chunk.metadata.items():
                if isinstance(value, (str, int, float, bool)):
                    clean_metadata[key] = value
            chunk.metadata = clean_metadata
    
        # ============ Save to Cache ============
        # CHUNKS_CACHE[source_type] = splits
        # ============ NEW: Save to Global Cache ============
        cache_data = {
            "splits": splits,
            "source_type": source_type,
            "newly_added_files": newly_added_files,
            "already_ingested": already_ingested,
            "ingest_state_path": ingest_state_path,
            "vectorstore_dir_abs": vectorstore_dir_abs
        }
        try:
            with open(CHUNK_CACHE_PATH, "wb") as f:
                pickle.dump(cache_data, f)
            logger.info(f"💾 Saved {len(splits)} chunks and file state to {CHUNK_CACHE_PATH}.")
        except Exception as e:
            logger.error(f"⚠️ Failed to save cache: {e}")

    # ============ FIX: Batching the Ingestion ============
    # Old Chroma.from_documents(...) acts as a shortcut. It attempts to create the database connection, 
    # initialize the embedding model, and ingest all the documents in a single, monolithic operation.
    # The new code splits this up. It first creates an empty connection to the database. 
    # This gives us an object (vectorstore) that we can feed data into slowly.
    logger.info("Initializing Chroma vectorstore and generating embeddings...")
    try:
        # Chroma.from_documents(
        #     documents=splits,
        #     embedding=embeddings,
        #     persist_directory=vectorstore_dir
        # )
        
        # NEW CODE: Connect to DB first, insert documents later
        vectorstore = Chroma(
            persist_directory=str(vectorstore_dir_abs), 
            embedding_function=embeddings
        )

        # If the chunk is relatively huge, the code tried to pass all (like 28,262) text chunks into 
        # the database engine simultaneously. While LangChain tries to handle this under the hood, 
        # passing that much data to the SQLite compaction engine at once often causes memory overflows 
        # or write-timeouts, resulting in the error:
        # chromadb.errors.InternalError: Error in compaction: Failed to apply logs to the metadata segment.

        # The new code uses a for loop to slice massive list of chunks into smaller, digestible 
        # arrays of 2,000 chunks each. It uses vectorstore.add_documents() to insert them one batch at 
        # a time, allowing the database to safely commit the logs to the disk before moving to the next batch.

        # NEW CODE: Feeds the DB batch_size
        BATCH_SIZE = batch_size
        total_batches = (len(splits) // BATCH_SIZE) + 1

        # ============ NEW: Setup Checkpoint File ============
        checkpoint_path = os.path.join(vectorstore_dir_abs, f"checkpoint_{source_type}.json")
        start_index = 0

        # Check if a previous crash left a checkpoint behind
        if os.path.exists(checkpoint_path):
            try:
                with open(checkpoint_path, "r", encoding="utf-8") as f:
                    checkpoint_data = json.load(f)
                    start_index = checkpoint_data.get("last_processed_index", 0)
                logger.info(f"🔄 Recovered from crash! Resuming Chroma ingestion at chunk index {start_index}.")
            except Exception as e:
                logger.warning(f"⚠️ Failed to read checkpoint file, starting batching from the beginning. Reason: {e}")
        
        for i in range(start_index, len(splits), BATCH_SIZE):
            batch = splits[i : i + BATCH_SIZE]
            current_batch_num = (i // BATCH_SIZE) + 1
            logger.info(f"Upserting batch {current_batch_num} of {total_batches} into Chroma...")
            # Usage
            resources = get_resources()
            logger.info(resources)
            vectorstore.add_documents(batch)
            
            # ============ NEW: Save Checkpoint After Each Batch ============
            try:
                with open(checkpoint_path, "w", encoding="utf-8") as f:
                    # Save the index of the NEXT batch to start from
                    json.dump({"last_processed_index": i + BATCH_SIZE}, f)
                    logger.info(f"Checkpoint saved. Last processed index: {i + BATCH_SIZE} (i + BATCH_SIZE)")
            except Exception as e:
                logger.error(f"⚠️ Failed to write backup checkpoint at index {i + BATCH_SIZE}: {e}")

        # Update tracking state only after ALL batches finish
        updated_files = sorted(already_ingested.union(newly_added_files))
        with open(ingest_state_path, "w", encoding="utf-8") as f:
            json.dump({"vectorstore_dir": vectorstore_dir_abs, "files": updated_files}, f, ensure_ascii=False, indent=2)

        # ============ NEW: Clean Up Checkpoint File ============
        # Everything succeeded! Delete the checkpoint file to save space.
        if os.path.exists(checkpoint_path): os.remove(checkpoint_path)
        
        logger.info(f"✅ Successfully ingested {len(splits)} chunks into Chroma DB at {vectorstore_dir} for source: {source_type}")
        logger.info(f"Updated ingest tracking file: {ingest_state_path}")
        return True
    except Exception as e:
        logger.error(f"❌ Error during Chroma DB creation: {e}", exc_info=True)
        raise e

In [25]:
# # ==================================== Ingest PDFs (ONE-TIME) ====================================
# # Cache code version 1
# def ingest_embeddings_with_cache(docs_path, embeddings, vectorstore_dir, docs_type, source_type, batch_size=1000):

#     docs_path_abs = os.path.abspath(docs_path)
#     vectorstore_dir_abs = os.path.abspath(vectorstore_dir)
#     ingest_state_path = os.path.join(docs_path_abs, "ingested_files.json")

#     # 1. Define where you want to save the cache locally (No Global Variables)
#     CHUNK_CACHE_PATH = os.path.join(BASE_DIR, f"chunks_cache_{source_type}.pkl")

#     logger.info(f"Starting {source_type} ingestion from: {docs_path}")

#     # ==========================================
#     # 1. ATTEMPT TO LOAD THE PARSING CACHE FROM DISK
#     # ==========================================
#     cache_loaded = False
    
#     if os.path.exists(CHUNK_CACHE_PATH):
#         logger.info(f"📦 Found local chunk cache for {source_type}! Loading from disk...")
#         try:
#             with open(CHUNK_CACHE_PATH, "rb") as f:
#                 cache_data = pickle.load(f)
            
#             splits = cache_data["splits"]
#             newly_added_files = cache_data["newly_added_files"]
#             already_ingested = cache_data["already_ingested"]
            
#             logger.info(f"✅ Successfully loaded {len(splits)} chunks from cache. Skipping PDF parsing!")
#             cache_loaded = True
#         except Exception as e:
#             logger.warning(f"⚠️ Failed to load cache: {e}. Will parse files from scratch.")
#             cache_loaded = False

#     # ==========================================
#     # 2. PARSE AND SPLIT (Only if cache is empty/failed)
#     # ==========================================
#     if not cache_loaded:
#         if docs_type not in LOADER_DICT:
#             logger.error(f"Unsupported document type: {docs_type}. Supported types are: {list(LOADER_DICT.keys())}")
#             return False
        
#         # State is tied to vectorstore_dir. If DB path changes, treat as a fresh DB.
#         ingest_state = {"vectorstore_dir": vectorstore_dir_abs, "files": []}
    
#         if os.path.exists(ingest_state_path):
#             try:
#                 with open(ingest_state_path, "r", encoding="utf-8") as f:
#                     existing_state = json.load(f)
#                 if isinstance(existing_state, dict) and existing_state.get("vectorstore_dir") == vectorstore_dir_abs:
#                     ingest_state = {
#                         "vectorstore_dir": vectorstore_dir_abs,
#                         "files": existing_state.get("files", []) if isinstance(existing_state.get("files", []), list) else []
#                     }
#                 else:
#                     logger.info("Detected a new embedding DB path. Resetting ingest tracking state.")
#             except Exception as e:
#                 logger.warning(f"Failed to read ingest state file, starting fresh. Reason: {e}")
    
#         already_ingested = set(ingest_state["files"])
    
#         loader = DirectoryLoader(
#             str(docs_path),
#             glob=f"**/*.{docs_type}",
#             loader_cls=LOADER_DICT.get(docs_type),
#             silent_errors=False,
#             show_progress=True
#         )
    
#         logger.info("Loading documents (this may take a while)...")
#         try:
#             docs = loader.load()
#             logger.info(f"Successfully loaded {len(docs)} {docs_type.upper()} files.")
#         except Exception as e:
#             logger.error(f"❌ Failed to load {docs_type.upper()}s: {e}", exc_info=True)
#             return False
    
#         docs_to_ingest = []
#         newly_added_files = set()
#         # for doc in docs:
#         #     source_file = os.path.abspath(doc.metadata.get("source", ""))
#         #     if source_file and source_file in already_ingested:
#         #         continue
#         #     docs_to_ingest.append(doc)
#         #     if source_file:
#         #         newly_added_files.add(source_file)
    
#         # ============ Fix: Use relative path record ===============
#         for doc in docs:
#             raw_source = doc.metadata.get("source", "")
            
#             if raw_source:
#                 # 1. Get the absolute path of the file
#                 abs_source = os.path.abspath(raw_source)
                
#                 # 2. Find the relative path from your project root (BASE_DIR)
#                 # This turns '/data/Docs/OfficialDocs/file.pdf' into 'Docs/OfficialDocs/file.pdf'
#                 # Docs/OfficialDocs/file.pdf is better than /Docs/OfficialDocs/file.pdf
#                 rel_source = os.path.relpath(abs_source, start=BASE_DIR)
                
#                 # 3. Force forward slashes for Windows/Linux cross-compatibility
#                 source_file = rel_source.replace("\\", "/")
#             else:
#                 source_file = ""
                
#             if source_file and source_file in already_ingested:
#                 continue
                
#             docs_to_ingest.append(doc)
            
#             if source_file:
#                 newly_added_files.add(source_file)
        
#         if not docs_to_ingest:
#             logger.info("No new files to ingest. All matching files already exist in ingest state.")
#             return True
    
#         logger.info(f"New files to ingest: {len(docs_to_ingest)}")
    
#         logger.info("Splitting documents into chunks...")
#         text_splitter = RecursiveCharacterTextSplitter(chunk_size=2000, chunk_overlap=400)
#         splits = text_splitter.split_documents(docs_to_ingest)
#         logger.info(f"Created {len(splits)} text chunks.")
    
#         # Add source metadata to each chunk for better traceability if source_type is provided
#         if source_type:
#             for chunk in splits:
#                 chunk.metadata["source_type"] = source_type
#             logger.info(f"Added source_type metadata to each chunk: {source_type}")
    
#         # ============ FIX: Sanitize Metadata ============
#         # PyPDFLoader sometimes extracts complex objects (like nested dictionaries, empty lists, or None values). 
#         # Chroma's strict Rust/SQLite backend crashes violently if it encounters anything other than a string, 
#         # integer, float, or boolean in the metadata. The Fix: Added a loop before the database insertion 
#         # that strips out any invalid data types.
#         logger.info("Sanitizing metadata...")
#         for chunk in splits:
#             clean_metadata = {}
#             for key, value in chunk.metadata.items():
#                 if isinstance(value, (str, int, float, bool)):
#                     clean_metadata[key] = value
#             chunk.metadata = clean_metadata
    
#         # ============ Save to Global Cache ============
#         # GLOBAL_CHUNKS_CACHE[source_type] = splits
#         # ============ NEW: Save to Global Cache ============
#         cache_data = {
#             "splits": splits,
#             "source_type": source_type,
#             "newly_added_files": newly_added_files,
#             "already_ingested": already_ingested,
#             "ingest_state_path": ingest_state_path,
#             "vectorstore_dir_abs": vectorstore_dir_abs
#         }
        
#         # Save directly to disk
#         try:
#             with open(CHUNK_CACHE_PATH, "wb") as f:
#                 pickle.dump(cache_data, f)
#             logger.info(f"💾 Saved {len(splits)} chunks and file state directly to {CHUNK_CACHE_PATH}.")
#         except Exception as e:
#             logger.error(f"⚠️ Failed to save cache: {e}")

#     # ============ THE ELEGANT BACKUP FIX: Local Embedding Cache ============
#     # Store embeddings in a separate folder so they survive a DB crash
#     # Choose a path outside the vectorstore_dir so it isn't deleted on rollback
#     cache_dir = os.path.join(BASE_DIR, "embedding_cache", source_type)
#     os.makedirs(cache_dir, exist_ok=True)
#     logger.info(f"Embedding DB cache path: {cache_dir}")
#     store = LocalFileStore(cache_dir)
    
#     # Wrap your existing 'embeddings' model
#     # (Use a namespace so you don't mix up different models like OpenAI vs Cohere)
#     cached_embedder = CacheBackedEmbeddings.from_bytes_store(
#         underlying_embeddings=embeddings, 
#         document_embedding_cache=store, 
#         namespace=source_type
#     )

#     # ============ Protects the OLD data ============
#     db_backup_path = f"{vectorstore_dir_abs}_backup"
#     # If a database already exists, create a snapshot before we touch it
#     if os.path.exists(vectorstore_dir_abs):
#         logger.info("Existing database detected. Creating a pre-update snapshot to protect old data...")
#         if os.path.exists(db_backup_path):
#             shutil.rmtree(db_backup_path)
#         shutil.copytree(vectorstore_dir_abs, db_backup_path)
    
#     # ============ FIX: Batching the Ingestion ============
#     # Old Chroma.from_documents(...) acts as a shortcut. It attempts to create the database connection, 
#     # initialize the embedding model, and ingest all the documents in a single, monolithic operation.
#     # The new code splits this up. It first creates an empty connection to the database. 
#     # This gives us an object (vectorstore) that we can feed data into slowly.
#     logger.info("Initializing Chroma vectorstore and generating embeddings...")
#     try:
#         # Chroma.from_documents(
#         #     documents=splits,
#         #     embedding=embeddings,
#         #     persist_directory=vectorstore_dir
#         # )
        
#         # Pass the CACHED embedder into Chroma
#         vectorstore = Chroma(
#             persist_directory=str(vectorstore_dir_abs), 
#             embedding_function=cached_embedder
#         )

#         # If the chunk is relatively huge, the code tried to pass all (like 28,262) text chunks into 
#         # the database engine simultaneously. While LangChain tries to handle this under the hood, 
#         # passing that much data to the SQLite compaction engine at once often causes memory overflows 
#         # or write-timeouts, resulting in the error:
#         # chromadb.errors.InternalError: Error in compaction: Failed to apply logs to the metadata segment.

#         # The new code uses a for loop to slice massive list of chunks into smaller, digestible 
#         # arrays of 2,000 chunks each. It uses vectorstore.add_documents() to insert them one batch at 
#         # a time, allowing the database to safely commit the logs to the disk before moving to the next batch.

#         # NEW CODE: Feeds the DB 2000 chunks at a time
#         BATCH_SIZE = batch_size
#         total_batches = (len(splits) // BATCH_SIZE) + 1

#         for i in range(0, len(splits), BATCH_SIZE):
#             batch = splits[i : i + BATCH_SIZE]
#             current_batch_num = (i // BATCH_SIZE) + 1
#             logger.info(f"Upserting batch {current_batch_num} of {total_batches} into Chroma...")
#             # Usage
#             resources = get_resources()
#             logger.info(resources)
            
#             # If first run: calls API, saves to cache, inserts to DB.
#             # If retry after crash: loads from cache instantly, inserts to DB.
#             vectorstore.add_documents(batch)

#         # Update tracking state only after ALL batches finish
#         updated_files = sorted(already_ingested.union(newly_added_files))
#         with open(ingest_state_path, "w", encoding="utf-8") as f:
#             json.dump({"vectorstore_dir": vectorstore_dir_abs, "files": updated_files}, f, ensure_ascii=False, indent=2)

#         # ============ 3. CLEANUP ============
#         # If everything succeeds, we don't need the physical snapshot anymore
#         if os.path.exists(db_backup_path):
#             shutil.rmtree(db_backup_path)
#             logger.info("🗑️ Removed physical snapshot (update complete).")
        
#         logger.info(f"✅ Successfully ingested {len(splits)} chunks into Chroma DB at {vectorstore_dir} for source: {source_type}")
#         logger.info(f"Updated ingest tracking file: {ingest_state_path}")
#         return True
#     except Exception as e:
#         logger.error(f"❌ Fatal crash during ingestion: {e}")
        
#         # ============ CLEAN, AUTOMATED RECOVERY ============
#         logger.warning("⚠️ Wiping corrupted database. Embeddings are safely cached on disk. Ready to rerun.")
        
#         # Free the file lock
#         if 'vectorstore' in locals(): del vectorstore

#         # Nuke the corrupted database. We don't care, we have the cache!
#         if os.path.exists(vectorstore_dir_abs): shutil.rmtree(vectorstore_dir_abs)

#         # Restore the old data from the physical snapshot!
#         if os.path.exists(db_backup_path):
#             shutil.copytree(db_backup_path, vectorstore_dir_abs)
#             logger.info("✅ Database successfully restored to its original state. Ready to safely rerun.")
#         raise e

In [8]:
# ==================================== Ingest PDFs (ONE-TIME) ====================================
# Cache code version 2 (add unique id)
def ingest_embeddings_with_cache(docs_path, embeddings, vectorstore_dir, docs_type, source_type, batch_size=1000):

    docs_path_abs = os.path.abspath(docs_path)
    vectorstore_dir_abs = os.path.abspath(vectorstore_dir)
    ingest_state_path = os.path.join(docs_path_abs, "ingested_files.json")

    # 1. Define where you want to save the cache locally (No Global Variables)
    CHUNK_CACHE_PATH = os.path.join(BASE_DIR, f"chunks_cache_{source_type}.pkl")

    logger.info(f"Starting {source_type} ingestion from: {docs_path}")

    # ==========================================
    # 1. ATTEMPT TO LOAD THE PARSING CACHE FROM DISK
    # ==========================================
    cache_loaded = False
    
    if os.path.exists(CHUNK_CACHE_PATH):
        logger.info(f"📦 Found local chunk cache for {source_type}! Loading from disk...")
        try:
            with open(CHUNK_CACHE_PATH, "rb") as f:
                cache_data = pickle.load(f)
            
            splits = cache_data["splits"]
            newly_added_files = cache_data["newly_added_files"]
            already_ingested = cache_data["already_ingested"]
            
            logger.info(f"✅ Successfully loaded {len(splits)} chunks from cache. Skipping PDF parsing!")
            cache_loaded = True
        except Exception as e:
            logger.warning(f"⚠️ Failed to load cache: {e}. Will parse files from scratch.")
            cache_loaded = False

    # ==========================================
    # 2. PARSE AND SPLIT (Only if cache is empty/failed)
    # ==========================================
    if not cache_loaded:
        if docs_type not in LOADER_DICT:
            logger.error(f"Unsupported document type: {docs_type}. Supported types are: {list(LOADER_DICT.keys())}")
            return False
        
        # State is tied to vectorstore_dir. If DB path changes, treat as a fresh DB.
        ingest_state = {"vectorstore_dir": vectorstore_dir_abs, "files": []}
    
        if os.path.exists(ingest_state_path):
            try:
                with open(ingest_state_path, "r", encoding="utf-8") as f:
                    existing_state = json.load(f)
                if isinstance(existing_state, dict) and existing_state.get("vectorstore_dir") == vectorstore_dir_abs:
                    ingest_state = {
                        "vectorstore_dir": vectorstore_dir_abs,
                        "files": existing_state.get("files", []) if isinstance(existing_state.get("files", []), list) else []
                    }
                else:
                    logger.info("Detected a new embedding DB path. Resetting ingest tracking state.")
            except Exception as e:
                logger.warning(f"Failed to read ingest state file, starting fresh. Reason: {e}")
    
        already_ingested = set(ingest_state["files"])
    
        loader = DirectoryLoader(
            str(docs_path),
            glob=f"**/*.{docs_type}",
            loader_cls=LOADER_DICT.get(docs_type),
            silent_errors=False,
            show_progress=True
        )
    
        logger.info("Loading documents (this may take a while)...")
        try:
            docs = loader.load()
            logger.info(f"Successfully loaded {len(docs)} {docs_type.upper()} files.")
        except Exception as e:
            logger.error(f"❌ Failed to load {docs_type.upper()}s: {e}", exc_info=True)
            return False
    
        docs_to_ingest = []
        newly_added_files = set()
        # for doc in docs:
        #     source_file = os.path.abspath(doc.metadata.get("source", ""))
        #     if source_file and source_file in already_ingested:
        #         continue
        #     docs_to_ingest.append(doc)
        #     if source_file:
        #         newly_added_files.add(source_file)
    
        # ============ Fix: Use relative path record ===============
        for doc in docs:
            raw_source = doc.metadata.get("source", "")
            
            if raw_source:
                # 1. Get the absolute path of the file
                abs_source = os.path.abspath(raw_source)
                
                # 2. Find the relative path from your project root (BASE_DIR)
                # This turns '/data/Docs/OfficialDocs/file.pdf' into 'Docs/OfficialDocs/file.pdf'
                # Docs/OfficialDocs/file.pdf is better than /Docs/OfficialDocs/file.pdf
                rel_source = os.path.relpath(abs_source, start=BASE_DIR)
                
                # 3. Force forward slashes for Windows/Linux cross-compatibility
                source_file = rel_source.replace("\\", "/")
            else:
                source_file = ""
                
            if source_file and source_file in already_ingested:
                continue
                
            docs_to_ingest.append(doc)
            
            if source_file:
                newly_added_files.add(source_file)
        
        if not docs_to_ingest:
            logger.info("No new files to ingest. All matching files already exist in ingest state.")
            return True
    
        logger.info(f"New files to ingest: {len(docs_to_ingest)}")
    
        logger.info("Splitting documents into chunks...")
        text_splitter = RecursiveCharacterTextSplitter(chunk_size=2000, chunk_overlap=400)
        splits = text_splitter.split_documents(docs_to_ingest)
        logger.info(f"Created {len(splits)} text chunks.")
    
        # Add source metadata to each chunk for better traceability if source_type is provided
        if source_type:
            for chunk in splits:
                chunk.metadata["source_type"] = source_type
            logger.info(f"Added source_type metadata to each chunk: {source_type}")
    
        # ============ FIX: Sanitize Metadata ============
        # PyPDFLoader sometimes extracts complex objects (like nested dictionaries, empty lists, or None values). 
        # Chroma's strict Rust/SQLite backend crashes violently if it encounters anything other than a string, 
        # integer, float, or boolean in the metadata. The Fix: Added a loop before the database insertion 
        # that strips out any invalid data types.
        logger.info("Sanitizing metadata...")
        for chunk in splits:
            clean_metadata = {}
            for key, value in chunk.metadata.items():
                if isinstance(value, (str, int, float, bool)):
                    clean_metadata[key] = value
            chunk.metadata = clean_metadata
        
        # ============ Save to Global Cache ============
        # GLOBAL_CHUNKS_CACHE[source_type] = splits
        # ============ NEW: Save to Global Cache ============
        cache_data = {
            "splits": splits,
            "source_type": source_type,
            "newly_added_files": newly_added_files,
            "already_ingested": already_ingested,
            "ingest_state_path": ingest_state_path,
            "vectorstore_dir_abs": vectorstore_dir_abs
        }
        
        # Save directly to disk
        try:
            with open(CHUNK_CACHE_PATH, "wb") as f:
                pickle.dump(cache_data, f)
            logger.info(f"💾 Saved {len(splits)} chunks and file state directly to {CHUNK_CACHE_PATH}.")
        except Exception as e:
            logger.error(f"⚠️ Failed to save cache: {e}")

    # ==========================================
    # GENERATE IDs (Runs every time!)
    # ==========================================
    # We do this here so it runs whether splits came from the .pkl or fresh parsing
    logger.info("Generating unique IDs for chunks...")
    chunk_ids = []
    for chunk in splits:
        unique_string = f"{chunk.metadata.get('source', '')}_{chunk.page_content}"
        chunk_hash = hashlib.sha256(unique_string.encode("utf-8")).hexdigest()
        chunk_ids.append(chunk_hash)
    
    # ============ THE ELEGANT BACKUP FIX: Local Embedding Cache ============
    # Store embeddings in a separate folder so they survive a DB crash
    # Choose a path outside the vectorstore_dir so it isn't deleted on rollback
    cache_dir = os.path.join(BASE_DIR, "embedding_cache", source_type)
    os.makedirs(cache_dir, exist_ok=True)
    logger.info(f"Embedding DB cache path: {cache_dir}")
    store = LocalFileStore(cache_dir)
    
    # Wrap your existing 'embeddings' model
    # (Use a namespace so you don't mix up different models like OpenAI vs Cohere)
    cached_embedder = CacheBackedEmbeddings.from_bytes_store(
        underlying_embeddings=embeddings, 
        document_embedding_cache=store, 
        namespace=source_type
    )
    
    # ============ FIX: Batching the Ingestion ============
    # Old Chroma.from_documents(...) acts as a shortcut. It attempts to create the database connection, 
    # initialize the embedding model, and ingest all the documents in a single, monolithic operation.
    # The new code splits this up. It first creates an empty connection to the database. 
    # This gives us an object (vectorstore) that we can feed data into slowly.
    logger.info("Initializing Chroma vectorstore and generating embeddings...")
    try:
        # Chroma.from_documents(
        #     documents=splits,
        #     embedding=embeddings,
        #     persist_directory=vectorstore_dir
        # )
        
        # Pass the CACHED embedder into Chroma
        vectorstore = Chroma(
            persist_directory=str(vectorstore_dir_abs), 
            embedding_function=cached_embedder
        )

        # If the chunk is relatively huge, the code tried to pass all (like 28,262) text chunks into 
        # the database engine simultaneously. While LangChain tries to handle this under the hood, 
        # passing that much data to the SQLite compaction engine at once often causes memory overflows 
        # or write-timeouts, resulting in the error:
        # chromadb.errors.InternalError: Error in compaction: Failed to apply logs to the metadata segment.

        # The new code uses a for loop to slice massive list of chunks into smaller, digestible 
        # arrays of 2,000 chunks each. It uses vectorstore.add_documents() to insert them one batch at 
        # a time, allowing the database to safely commit the logs to the disk before moving to the next batch.

        # NEW CODE: Feeds the DB 2000 chunks at a time
        BATCH_SIZE = batch_size
        total_batches = (len(splits) // BATCH_SIZE) + 1

        for i in range(0, len(splits), BATCH_SIZE):
            batch = splits[i : i + BATCH_SIZE]
            batch_ids = chunk_ids[i : i + BATCH_SIZE] # <-- Now this is safely defined!
            
            current_batch_num = (i // BATCH_SIZE) + 1
            logger.info(f"Upserting batch {current_batch_num} of {total_batches} into Chroma...")
            # Usage
            resources = get_resources()
            logger.info(resources)
            
            # If first run: calls API, saves to cache, inserts to DB.
            # If retry after crash: loads from cache instantly, inserts to DB.
            vectorstore.add_documents(documents=batch, ids=batch_ids)

        # Update tracking state only after ALL batches finish
        updated_files = sorted(already_ingested.union(newly_added_files))
        with open(ingest_state_path, "w", encoding="utf-8") as f:
            json.dump({"vectorstore_dir": vectorstore_dir_abs, "files": updated_files}, f, ensure_ascii=False, indent=2)

        logger.info(f"✅ Successfully ingested {len(splits)} chunks into Chroma DB at {vectorstore_dir} for source: {source_type}")
        logger.info(f"Updated ingest tracking file: {ingest_state_path}")
        return True
    except Exception as e:
        logger.error(f"❌ Crash during ingestion: {e}")
        logger.warning("⚠️ Script crashed! BUT previously successful batches are safely preserved in Chroma.")
        logger.warning("When you rerun this script, Chroma will skip the completed batches and resume work.")
        
        if 'vectorstore' in locals(): del vectorstore
        raise e

# Potential Bug

By pointing this out, you uncovered a flaw in my previous recovery instructions.

Imagine this scenario:

1. We backup the DB (Pre-ingestion state: **0 new batches**).
2. The loop successfully ingests Batch 1, 2, and 3. The checkpoint saves: **"Start at Batch 4"**.
3. During Batch 4, the system runs out of memory and the database violently **corrupts**.
4. You restore the physical backup folder.

**The Bug:** The restored database has **0 new batches** in it. But your `checkpoint.json` still says **"Start at Batch 4"**. If you run the script, it will skip batches 1, 2, and 3, and those chunks will be permanently missing from your database!

### The Bulletproof Solution

The elegant, native way to solve this is to **cache the embeddings directly to your hard drive**, completely separate from Chroma.

If we use LangChain’s `CacheBackedEmbeddings`, the workflow becomes incredibly simple:

1. The script hashes each chunk and checks a local folder.
2. If the embedding isn't there, it calls the API (costs money/time) and saves it to the folder.
3. If Chroma violently crashes at Batch 4, you just let the script delete the corrupted Chroma folder and start from Batch 1 again.
4. **The Magic:** When it re-runs Batches 1, 2, and 3, it sees the embeddings on your hard drive. It skips the API entirely, costing **$0** and taking **milliseconds**, before smoothly resuming Batch 4.

### The Elegant Code

Here is how beautifully simple the code becomes. No checkpoint files, no complex loop math, and no starting/stopping the database.

```python
import shutil
from langchain.storage import LocalFileStore
from langchain.embeddings import CacheBackedEmbeddings

    # ... [previous code: chunking and sanitizing metadata] ...

    # ============ THE ELEGANT FIX: Local Embedding Cache ============
    # Store embeddings in a separate folder so they survive a DB crash
    # Choose a path outside the vectorstore_dir so it isn't deleted on rollback
    cache_dir = os.path.join(BASE_DIR, "embedding_cache")
    store = LocalFileStore(cache_dir)
    
    # Wrap your existing 'embeddings' model
    # (Use a namespace so you don't mix up different models like OpenAI vs Cohere)
    cached_embedder = CacheBackedEmbeddings.from_bytes_store(
        underlying_embeddings=embeddings, 
        document_embedding_cache=store, 
        namespace="my_project_embeddings" 
    )

    logger.info("Initializing Chroma vectorstore...")
    try:
        # Pass the CACHED embedder into Chroma
        vectorstore = Chroma(
            persist_directory=str(vectorstore_dir_abs), 
            embedding_function=cached_embedder
        )

        BATCH_SIZE = 2000
        total_batches = (len(splits) // BATCH_SIZE) + 1

        # A simple, clean loop
        for i in range(0, len(splits), BATCH_SIZE):
            batch = splits[i : i + BATCH_SIZE]
            logger.info(f"Upserting batch {(i // BATCH_SIZE) + 1} of {total_batches} into Chroma...")
            
            # If first run: calls API, saves to cache, inserts to DB.
            # If retry after crash: loads from cache instantly, inserts to DB.
            vectorstore.add_documents(batch)

        # Update ingest state only when 100% successful
        updated_files = sorted(already_ingested.union(newly_added_files))
        with open(ingest_state_path, "w", encoding="utf-8") as f:
            json.dump({"vectorstore_dir": vectorstore_dir_abs, "files": updated_files}, f, ensure_ascii=False, indent=2)

        logger.info(f"✅ Successfully ingested {len(splits)} chunks into Chroma DB.")
        return True

    except Exception as e:
        logger.error(f"❌ Fatal crash during ingestion: {e}")
        
        # ============ CLEAN, AUTOMATED RECOVERY ============
        logger.warning("⚠️ Wiping corrupted database. Embeddings are safely cached on disk. Ready to rerun.")
        
        # Free the file lock
        if 'vectorstore' in locals():
            del vectorstore
            
        # Nuke the corrupted database. We don't care, we have the cache!
        if os.path.exists(vectorstore_dir_abs):
            shutil.rmtree(vectorstore_dir_abs)
            
        raise e

```

### Why this is the best architecture:

* **Separation of Concerns:** Chroma is fragile, so we treat it as disposable. Your embeddings are valuable, so we treat them as permanent.
* **Cleaner Code:** You don't have to manage state or manipulate `start_index` integers.
* **Future Proofing:** If you ever decide to ditch Chroma and switch to FAISS or Qdrant later, you *already* have all your embeddings saved to your hard drive. You won't have to pay for them again.

# Why works?

You are exactly right—there is absolutely no database backup code in the loop anymore. That is the magic of this approach.

To understand why this works, we have to change how we think about the problem. In the previous attempts, we were trying to save the **Chroma Database**. But the database itself isn't what is valuable; Chroma is just a free, local SQLite file.

What is actually valuable (what costs you time and API money) are the **Embeddings**—the mathematical vectors generated by your embedding model.

Here is exactly how the `CacheBackedEmbeddings` acts as an invisible, automatic backup system without you having to write any backup code in the loop.

### How the Invisible Backup Works

When you call `vectorstore.add_documents(batch)`, you are actually triggering a chain reaction. Because we wrapped your embedding model in LangChain's `CacheBackedEmbeddings`, this happens during **Batch 1**:

1. **The Hash:** LangChain looks at the text in Batch 1 and creates a unique ID (a hash) for it.
2. **The Hard Drive Check:** It checks the `embedding_cache` folder on your hard drive to see if that ID exists.
3. **The "Backup" (First Run):** Because it is the first run, the folder is empty. LangChain sends the text to your embedding model (costing time/money). As soon as the model returns the vectors, LangChain **saves those vectors as a file in your `embedding_cache` folder**.
4. **The DB Insert:** Finally, the vectors are handed to Chroma to be inserted into the database.

LangChain handles that file-saving process natively. That *is* your backup. It happens automatically inside `add_documents`.

### The Crash & Recovery Scenario

Let's look at the exact scenario you brought up earlier (the "Dirty Crash" at Batch 4) and see how the cache solves it elegantly:

**The Crash:**

1. Batches 1, 2, and 3 succeed. (Their vectors are now permanently saved in the `embedding_cache` folder on your hard drive).
2. **Batch 4 Causes a Dirty Crash.** Chroma violently corrupts.
3. The `except` block fires, catches the error, and simply **deletes the entire corrupted Chroma database**.

**The Recovery (The Magic):**

1. You run the script again. The script creates a brand new, empty Chroma database.
2. The loop starts at **Batch 1**. (No checkpoint file needed).
3. LangChain looks at Batch 1, hashes it, and checks the `embedding_cache` folder.
4. **The Catch:** It finds the vectors sitting right there on your hard drive!
5. Instead of calling the API, it instantly loads the vectors from your hard drive and inserts them into Chroma.

Re-running Batches 1, 2, and 3 takes just a few milliseconds and costs $0 because the API is bypassed entirely. The loop smoothly catches up to Batch 4, calls the API for the new data, saves Batch 4 to the cache folder, and keeps going.

### Why this is better than copying the Database

By moving the "backup" away from the fragile SQLite database and applying it directly to the raw embedding vectors, you eliminate all the complex file-locking, the risk of data mismatch, and the heavy disk I/O of copying gigabytes of folders. You just let the database die, build a new one, and fast-forward through the cached data.

---

# Error
```python
[26-05-10 19:34:30][ERROR] ❌ Error during Chroma DB creation: Error in compaction: Failed to apply logs to the metadata segment
Traceback (most recent call last):
  File "/tmp/ipykernel_1871/918331308.py", line 170, in ingest_embeddings
    vectorstore.add_documents(batch)
  File "/home/user/miniconda/lib/python3.12/site-packages/langchain_core/vectorstores/base.py", line 258, in add_documents
    return self.add_texts(texts, metadatas, **kwargs)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/home/user/miniconda/lib/python3.12/site-packages/langchain_chroma/vectorstores.py", line 651, in add_texts
    self._collection.upsert(
  File "/home/user/miniconda/lib/python3.12/site-packages/chromadb/api/models/Collection.py", line 530, in upsert
    self._client._upsert(
  File "/home/user/miniconda/lib/python3.12/site-packages/chromadb/api/rust.py", line 541, in _upsert
    return self.bindings.upsert(
           ^^^^^^^^^^^^^^^^^^^^^
chromadb.errors.InternalError: Error in compaction: Failed to apply logs to the metadata segment
---------------------------------------------------------------------------
InternalError                             Traceback (most recent call last)
Cell In[21], line 1
----> 1 Success_Forum_China = ingest_embeddings(WQB_FORUM_CHINA_PATH, embeddings, EMBEDDING_DB_FORUM_CHINA_DIR, "pdf", "forum_china_pdf")

Cell In[7], line 181, in ingest_embeddings(docs_path, embeddings, vectorstore_dir, docs_type, source_type)
    177         logger.info(f"Updated ingest tracking file: {ingest_state_path}")
    178         return True
    179     except Exception as e:
    180         logger.error(f"❌ Error during Chroma DB creation: {e}", exc_info=True)
--> 181         raise e

Cell In[7], line 181, in ingest_embeddings(docs_path, embeddings, vectorstore_dir, docs_type, source_type)
    177         logger.info(f"Updated ingest tracking file: {ingest_state_path}")
    178         return True
    179     except Exception as e:
    180         logger.error(f"❌ Error during Chroma DB creation: {e}", exc_info=True)
--> 181         raise e

File ~/miniconda/lib/python3.12/site-packages/langchain_core/vectorstores/base.py:258, in VectorStore.add_documents(self, documents, **kwargs)
    256     texts = [doc.page_content for doc in documents]
    257     metadatas = [doc.metadata for doc in documents]
--> 258     return self.add_texts(texts, metadatas, **kwargs)
    259 msg = (
    260     f"`add_documents` and `add_texts` has not been implemented "
    261     f"for {self.__class__.__name__} "
    262 )
    263 raise NotImplementedError(msg)

File ~/miniconda/lib/python3.12/site-packages/langchain_chroma/vectorstores.py:651, in Chroma.add_texts(self, texts, metadatas, ids, **kwargs)
    649 ids_with_metadata = [ids[idx] for idx in non_empty_ids]
    650 try:
--> 651     self._collection.upsert(
    652         metadatas=metadatas,  # type: ignore[arg-type]
    653         embeddings=embeddings_with_metadatas,  # type: ignore[arg-type]
    654         documents=texts_with_metadatas,
    655         ids=ids_with_metadata,
    656     )
    657 except ValueError as e:
    658     if "Expected metadata value to be" in str(e):

File ~/miniconda/lib/python3.12/site-packages/chromadb/api/models/Collection.py:530, in Collection.upsert(self, ids, embeddings, metadatas, documents, images, uris)
    511 """Create or update records by ID.
    512 
    513 Args:
   (...)    519     uris: URIs for loading images.
    520 """
    521 upsert_request = self._validate_and_prepare_upsert_request(
    522     ids=ids,
    523     embeddings=embeddings,
   (...)    527     uris=uris,
    528 )
--> 530 self._client._upsert(
    531     collection_id=self.id,
    532     ids=upsert_request["ids"],
    533     embeddings=upsert_request["embeddings"],
    534     metadatas=upsert_request["metadatas"],
    535     documents=upsert_request["documents"],
    536     uris=upsert_request["uris"],
    537     tenant=self.tenant,
    538     database=self.database,
    539 )

File ~/miniconda/lib/python3.12/site-packages/chromadb/api/rust.py:541, in RustBindingsAPI._upsert(self, collection_id, ids, embeddings, metadatas, documents, uris, tenant, database)
    529 @override
    530 def _upsert(
    531     self,
   (...)    539     database: str = DEFAULT_DATABASE,
    540 ) -> bool:
--> 541     return self.bindings.upsert(
    542         str(collection_id),
    543         ids,
    544         embeddings,
    545         metadatas,
    546         documents,
    547         uris,
    548         tenant,
    549         database,
    550     )

InternalError: Error in compaction: Failed to apply logs to the metadata segment
```

This specific error—`Failed to apply logs to the metadata segment`—is the database equivalent of choking on your food because you tried to swallow too much at once.

To understand why this happened, you have to look at how ChromaDB saves data under the hood. It doesn't write your documents directly to the permanent database file. Instead, it uses a two-step process:

1. **The Log (Fast):** When you call `add_documents(batch)`, Chroma quickly writes that batch to a temporary "Write-Ahead Log" (WAL).
2. **The Compaction (Heavy):** Once that log gets full, Chroma triggers a background process called **compaction**. It takes all that temporary data, heavily processes it, and merges it into the permanent SQLite metadata file and the binary vector files.

Your script crashed right in the middle of Step 2. Here are the three most common reasons why the compaction process violently fails:

### 1. You Ran Out of RAM (Memory Spike)

This is the #1 culprit for this exact error. Compacting a database is a highly memory-intensive operation. Chroma has to load the existing metadata segment into your system's RAM, merge the new 2,000 chunks into it, and write it back out. If your computer (or Docker container/cloud server) hits its memory limit during this process, the operating system instantly kills the compaction thread to protect itself. The database is left fractured.

### 2. Disk I/O Timeout (Slow Hard Drive)

If you are running this on a slower hard drive (HDD), a networked drive, or a cloud server with restricted disk write speeds, the compaction process might simply take too long. SQLite expects data to be written within a specific timeframe. If the disk is struggling to write the massive surge of data, the database engine assumes it's broken, times out, and throws this internal error.

### 3. File Lock Conflicts

Sometimes, the temporary log file gets "locked" by another background thread in Python, or by your operating system's antivirus scanner deciding to scan the newly created file. When the compaction engine tries to read that log file to merge it, it gets blocked by the lock and crashes.

---

### How to Prevent This from Happening Again

Since the caching system we discussed earlier will protect your embeddings and let you safely recover, your next goal is to stop the crash from happening in the first place.

**1. Lower your `BATCH_SIZE`**
2,000 chunks was a good starting guess, but your specific hardware is telling you it's too heavy. Drop it down to give the compaction engine smaller, more digestible bites.

```python
# Change this:
BATCH_SIZE = 2000

# To this:
BATCH_SIZE = 500

```

This forces Chroma to do more frequent, but much smaller and safer compactions. It will dramatically lower your RAM usage and prevent disk timeouts.

**2. Monitor your RAM**
The next time you run the script, open your system monitor (Task Manager on Windows, or run `htop` / `top` in your Linux terminal). Watch your memory usage as the script runs. If you see your RAM bar maxing out right as the batch finishes, you know for a fact it's an Out-Of-Memory kill.

---

# Cache Note 🚨
For LangChain's CacheBackedEmbeddings, the script doesn't wait for the very end to save everything. It usually saves to the LocalFileStore as soon as a batch of embeddings is finished.

In [ ]:
Success_OfficialDocs = ingest_embeddings_no_cache(WQB_OFFICIAL_DOCS_PATH, embeddings, EMBEDDING_DB_OFFICIALDOCS_DIR, "pdf", "official_docs_pdf")

[26-05-10 12:56:50][INFO] Starting official_docs_pdf ingestion from: /data/Docs/OfficialDocs
[26-05-10 12:56:50][INFO] Loading documents (this may take a while)...
100%|██████████| 77/77 [00:09<00:00,  8.01it/s]
[26-05-10 12:57:00][INFO] Successfully loaded 238 PDF files.
[26-05-10 12:57:00][INFO] New files to ingest: 238
[26-05-10 12:57:00][INFO] Splitting documents into chunks...
[26-05-10 12:57:00][INFO] Created 290 text chunks.
[26-05-10 12:57:00][INFO] Added source_type metadata to each chunk: official_docs_pdf
[26-05-10 12:57:00][INFO] Sanitizing metadata...
[26-05-10 12:57:00][INFO] 💾 Saved 290 chunks and file state to GLOBAL_CHUNKS_CACHE['official_docs_pdf'] as a backup.
[26-05-10 12:57:00][INFO] Initializing Chroma vectorstore and generating embeddings...
[26-05-10 12:57:00][INFO] Upserting batch 1 of 1 into Chroma...


Batches:   0%|          | 0/10 [00:00<?, ?it/s]

[26-05-10 13:10:20][INFO] ✅ Successfully ingested 290 chunks into Chroma DB at /data/embedding_db/wqb_official_docs_embedding_db for source: official_docs_pdf
[26-05-10 13:10:20][INFO] Updated ingest tracking file: /data/Docs/OfficialDocs/ingested_files.json


In [12]:
# Reset cache variable
GLOBAL_CHUNKS_CACHE = {}

In [ ]:
# Usage
resources = get_resources()
print(resources)

In [ ]:
Success_Forum_Tips = ingest_embeddings_no_cache(WQB_FORUM_TIPS_PATH, embeddings, EMBEDDING_DB_FORUM_TIPS_DIR, "pdf", "forum_tips_pdf")

[26-05-10 13:22:06][INFO] Starting forum_tips_pdf ingestion from: /data/Docs/Forums/wqb_brain_tips_pdf
[26-05-10 13:22:06][INFO] Loading documents (this may take a while)...
 99%|█████████▉| 80/81 [00:44<00:00,  1.79it/s]
[26-05-10 13:22:51][INFO] Successfully loaded 283 PDF files.
[26-05-10 13:22:51][INFO] New files to ingest: 283
[26-05-10 13:22:51][INFO] Splitting documents into chunks...
[26-05-10 13:22:51][INFO] Created 350 text chunks.
[26-05-10 13:22:51][INFO] Added source_type metadata to each chunk: forum_tips_pdf
[26-05-10 13:22:51][INFO] Sanitizing metadata...
[26-05-10 13:22:51][INFO] 💾 Saved 350 chunks and file state to GLOBAL_CHUNKS_CACHE['forum_tips_pdf'] as a backup.
[26-05-10 13:22:51][INFO] Initializing Chroma vectorstore and generating embeddings...
[26-05-10 13:22:51][INFO] Upserting batch 1 of 1 into Chroma...


Batches:   0%|          | 0/11 [00:00<?, ?it/s]

In [ ]:
Success_Forum_Research = ingest_embeddings_no_cache(WQB_FORUM_RESEARCH_PATH, embeddings, EMBEDDING_DB_FORUM_RESEARCH_DIR, "pdf", "forum_research_pdf")

[26-05-10 13:50:37][INFO] Starting forum_research_pdf ingestion from: /data/Docs/Forums/wqb_research_pdf
[26-05-10 13:50:37][INFO] Loading documents (this may take a while)...
100%|██████████| 485/485 [02:28<00:00,  3.26it/s]
[26-05-10 13:53:06][INFO] Successfully loaded 957 PDF files.
[26-05-10 13:53:06][INFO] New files to ingest: 957
[26-05-10 13:53:06][INFO] Splitting documents into chunks...
[26-05-10 13:53:06][INFO] Created 1093 text chunks.
[26-05-10 13:53:06][INFO] Added source_type metadata to each chunk: forum_research_pdf
[26-05-10 13:53:06][INFO] Sanitizing metadata...
[26-05-10 13:53:06][INFO] 💾 Saved 1093 chunks and file state to GLOBAL_CHUNKS_CACHE['forum_research_pdf'] as a backup.
[26-05-10 13:53:06][INFO] Initializing Chroma vectorstore and generating embeddings...
[26-05-10 13:53:06][INFO] Upserting batch 1 of 1 into Chroma...


Batches:   0%|          | 0/35 [00:00<?, ?it/s]

In [13]:
Success_Forum_China = ingest_embeddings_with_cache(WQB_FORUM_CHINA_PATH, embeddings, EMBEDDING_DB_FORUM_CHINA_DIR, "pdf", "forum_china_pdf")

[26-05-10 20:32:27][INFO] Starting forum_china_pdf ingestion from: /data/Docs/Forums/wqb_china_consultant_pdf
[26-05-10 20:32:27][INFO] Loading documents (this may take a while)...
100%|██████████| 1398/1398 [22:24<00:00,  1.04it/s]
[26-05-10 20:55:10][INFO] Successfully loaded 11609 PDF files.
[26-05-10 20:55:11][INFO] New files to ingest: 11609
[26-05-10 20:55:11][INFO] Splitting documents into chunks...
[26-05-10 20:55:11][INFO] Created 12414 text chunks.
[26-05-10 20:55:11][INFO] Added source_type metadata to each chunk: forum_china_pdf
[26-05-10 20:55:11][INFO] Sanitizing metadata...
[26-05-10 20:55:12][INFO] 💾 Saved 12414 chunks and file state directly to /data/chunks_cache_forum_china_pdf.pkl.
/home/user/miniconda/lib/python3.12/site-packages/langchain_classic/embeddings/cache.py:58: UserWarning: Using default key encoder: SHA-1 is *not* collision-resistant. While acceptable for most cache scenarios, a motivated attacker can craft two different payloads that map to the same cach

Batches:   0%|          | 0/32 [00:00<?, ?it/s]

[26-05-10 21:43:36][INFO] Upserting batch 2 of 13 into Chroma...
[26-05-10 21:43:36][INFO] {'CPU': '55.6%', 'RAM': '89.8%', 'RAM_Used': '111.18 GB', 'RAM_Total': '123.79 GB', 'RAM_Available': '12.61 GB'}


Batches:   0%|          | 0/31 [00:00<?, ?it/s]

[26-05-10 22:31:02][INFO] Upserting batch 3 of 13 into Chroma...
[26-05-10 22:31:02][INFO] {'CPU': '69.8%', 'RAM': '90.0%', 'RAM_Used': '111.36 GB', 'RAM_Total': '123.79 GB', 'RAM_Available': '12.43 GB'}


...

[26-05-11 06:07:59][INFO] Upserting batch 12 of 13 into Chroma...
[26-05-11 06:07:59][INFO] {'CPU': '65.1%', 'RAM': '97.4%', 'RAM_Used': '120.62 GB', 'RAM_Total': '123.79 GB', 'RAM_Available': '3.17 GB'}


Batches:   0%|          | 0/29 [00:00<?, ?it/s]

[26-05-11 08:59:16][INFO] Upserting batch 13 of 13 into Chroma...
[26-05-11 08:59:16][INFO] {'CPU': '61.1%', 'RAM': '92.4%', 'RAM_Used': '114.36 GB', 'RAM_Total': '123.79 GB', 'RAM_Available': '9.43 GB'}


Batches:   0%|          | 0/13 [00:00<?, ?it/s]

[26-05-11 09:24:13][INFO] 🗑️ Removed physical snapshot (update complete).
[26-05-11 09:24:13][INFO] ✅ Successfully ingested 12414 chunks into Chroma DB at /data/embedding_db/wqb_forum_china_embedding_db for source: forum_china_pdf
[26-05-11 09:24:13][INFO] Updated ingest tracking file: /data/Docs/Forums/wqb_china_consultant_pdf/ingested_files.json


In [9]:
# Usage
resources = get_resources()
print(resources)

{'CPU': '60.1%', 'RAM': '98.9%', 'RAM_Used': '122.47 GB', 'RAM_Total': '123.79 GB', 'RAM_Available': '1.32 GB'}


In [10]:
Success_Forum_Global = ingest_embeddings_with_cache(WQB_FORUM_GLOBAL_PATH, embeddings, EMBEDDING_DB_FORUM_GLOBAL_DIR, "pdf", "forum_global_pdf")

[26-05-11 15:55:49][INFO] Starting forum_global_pdf ingestion from: /data/Docs/Forums/wqb_global_consultant_pdf
[26-05-11 15:55:49][INFO] 📦 Found local chunk cache for forum_global_pdf! Loading from disk...
[26-05-11 15:55:49][INFO] ✅ Successfully loaded 14405 chunks from cache. Skipping PDF parsing!
[26-05-11 15:55:49][INFO] Generating unique IDs for chunks...
[26-05-11 15:55:49][INFO] Embedding DB cache path: /data/embedding_cache/forum_global_pdf
/home/user/miniconda/lib/python3.12/site-packages/langchain_classic/embeddings/cache.py:58: UserWarning: Using default key encoder: SHA-1 is *not* collision-resistant. While acceptable for most cache scenarios, a motivated attacker can craft two different payloads that map to the same cache key. If that risk matters in your environment, supply a stronger encoder (e.g. SHA-256 or BLAKE2) via the `key_encoder` argument. If you change the key encoder, consider also creating a new cache, to avoid (the potential for) collisions with existing key

Batches:   0%|          | 0/32 [00:00<?, ?it/s]

[26-05-11 16:36:07][INFO] Upserting batch 2 of 15 into Chroma...
[26-05-11 16:36:08][INFO] {'CPU': '63.5%', 'RAM': '92.0%', 'RAM_Used': '113.94 GB', 'RAM_Total': '123.79 GB', 'RAM_Available': '9.85 GB'}


Batches:   0%|          | 0/31 [00:00<?, ?it/s]

[26-05-11 17:13:49][INFO] Upserting batch 3 of 15 into Chroma...
[26-05-11 17:13:50][INFO] {'CPU': '56.6%', 'RAM': '92.0%', 'RAM_Used': '113.89 GB', 'RAM_Total': '123.79 GB', 'RAM_Available': '9.90 GB'}


...

[26-05-11 23:20:05][INFO] Upserting batch 14 of 15 into Chroma...
[26-05-11 23:20:05][INFO] {'CPU': '58.9%', 'RAM': '91.8%', 'RAM_Used': '113.67 GB', 'RAM_Total': '123.79 GB', 'RAM_Available': '10.12 GB'}


Batches:   0%|          | 0/30 [00:00<?, ?it/s]

[26-05-11 23:47:39][INFO] Upserting batch 15 of 15 into Chroma...
[26-05-11 23:47:39][INFO] {'CPU': '93.0%', 'RAM': '92.3%', 'RAM_Used': '114.26 GB', 'RAM_Total': '123.79 GB', 'RAM_Available': '9.53 GB'}


Batches:   0%|          | 0/12 [00:00<?, ?it/s]

[26-05-11 23:57:08][INFO] ✅ Successfully ingested 14405 chunks into Chroma DB at /data/embedding_db/wqb_forum_global_embedding_db for source: forum_global_pdf
[26-05-11 23:57:08][INFO] Updated ingest tracking file: /data/Docs/Forums/wqb_global_consultant_pdf/ingested_files.json


In [ ]:
# print("🚀 Launching rescue operation from RAM cache...")
# rescue_cached_embeddings(
#     source_type="forum_pdf", 
#     embeddings=embeddings, 
#     vectorstore_dir=EMBEDDING_DB_DIR
# )

In [11]:
# Usage
resources = get_resources()
print(resources)

{'CPU': '76.0%', 'RAM': '94.7%', 'RAM_Used': '117.27 GB', 'RAM_Total': '123.79 GB', 'RAM_Available': '6.52 GB'}


In [16]:
# ==================================== Initialize Retriever (for querying) ====================================
vectorstore = Chroma(persist_directory=EMBEDDING_DB_FORUM_GLOBAL_DIR, embedding_function=embeddings)
retriever = vectorstore.as_retriever(search_kwargs={"k": 8})

In [17]:
# ====================== DOCS SEARCH TOOL ======================
@tool("retrieve_text_data")
def retrieve_text_data(query: str) -> str:
    """Fetches relevant text snippets based on a query.
    Input a highly specific financial concept or math operator (e.g., 'supply chain momentum', 'analyst revision').
    Returns text context to be used for answering user queries."""
    
    docs = retriever.invoke(query)
    return "\n\n---\n\n".join([doc.page_content for doc in docs])

# ==================================== DEBUG: Test if your docs PDFs are loaded ====================================
print("Testing retrieve_text_data tool...")

test_result = retrieve_text_data.run("alpha")

print("\n" + "="*80)
print("TOOL TEST RESULT:")
print(test_result)  # first 1500 chars
print("\n" + "="*80)
print(f"Length of returned text: {len(test_result)} characters")

Testing retrieve_text_data tool...
Using Tool: retrieve_text_data


Batches:   0%|          | 0/1 [00:00<?, ?it/s]


TOOL TEST RESULT:
Next › »
Didn't find what you were looking for?
N ew post
© 2026 WorldQuant BRAIN®. All Rights Reserved.
1 year ago
Constructing a strong alpha demands meticulous data processing, normalization, and refinement.
Maintaining a consistent data flow, converting raw price signals into meaningful ranks, and emphasizing
technical trends while filtering out fundamental noise helps generate a cleaner and more reliable alpha signal.
24
NP65801
1 year ago
Alpha-generating strategies requires more than just identifying potential opportunities in financial markets;
it demands a deep understanding of data processing and the ability to adjust for fundamental factors that
influence asset prices. Alpha is the excess return that a strategy generates beyond a benchmark or expected
return, and achieving superior alpha involves efficient data handling, signal generation, and continuous
refinement.
14

---

exploration. Neutralization is also key — without rigorous orthogonalization to re